# Testing the Data Journalist Agent (LM Studio / MLX Edition)

This notebook tests the `DataJournalistAgent` with pre-calculated dataframes and stricter prompting.

In [1]:
import pandas as pd
from src.salary_data.scraper import Scraper
from src.salary_data.agent import DataJournalistAgent
import os
from dotenv import load_dotenv

load_dotenv()

# 1. Fetch Data
scraper = Scraper()
df_net = scraper.get_cgecse_salaries(scraper.URL_TESTIGO_NETO)
df_ipc = scraper.get_ipc_indec()
df_cba_cbt = scraper.get_cba_cbt()

dfs_dict = {
    "net_salaries": df_net,
    "inflation_ipc": df_ipc,
    "poverty_lines": df_cba_cbt
}

print("Data loaded successfully.")

Data loaded successfully.


In [19]:
df_cba_cbt.columns

Index(['canasta_basica_alimentaria', 'inversa_coeficiente_engel',
       'canasta_basica_total', 'linea_indigencia', 'linea_pobreza'],
      dtype='object')

In [18]:
real_salaries = scraper.calculate_real_salary(df_net, df_ipc['infl_Nivel_general'])
start = real_salaries.loc["2023-09-01"]
end = real_salaries.iloc[-1]
losses = (end - start) / start * 100
losses.sort_values(ascending=False, inplace=True)
print(losses)

jurisdiction
Neuquén                          23.775286
Santa Cruz                       21.286062
Tierra del Fuego                  9.734994
Chaco                             0.091365
Formosa                          -1.751847
Chubut                           -4.045690
San Juan                         -4.878124
Corrientes                       -5.504230
Ciudad de Buenos Aires           -8.948093
Río Negro                       -13.566062
La Pampa                        -14.328955
La Rioja                        -15.475045
Jujuy                           -15.874418
Misiones                        -16.827246
Tucumán                         -20.951172
Promedio Ponderado (MG Total)   -22.039396
Catamarca                       -22.229479
Entre Ríos                      -24.821213
Santa Fe                        -25.686764
Córdoba                         -28.264645
Buenos Aires                    -30.445337
Mendoza                         -34.181931
Salta                           -34.49694

In [2]:
model_params = {
    "model": "ollama/qwen3.5:9b",
    "base_url": "localhost:11434",
    "temperature": 0.2
}

# model_params = {
#     "model": "openai/gpt-4o-mini",
#     "temperature": 0.2
# }

journalist = DataJournalistAgent(dfs_dict, model_params=model_params)

## Test Queries
The agent now knows that data goes up to 2025 and has a full list of provinces in its prompt.

In [ ]:
# Question 1
q1 = "Which provinces lost more purchasing power between November 2023 and today?"
response1 = journalist.query(q1)
print(response1['output'])



> Entering new AgentExecutor chain...
Thought: I need to analyze which provinces lost the most purchasing power between November 2023 and September 2025. Since the data is quarterly, I'll use December 2023 as the closest date to November 2023, and September 2025 as "today". I should first calculate purchasing power loss for all provinces to compare them.

Action: calculate_purchasing_power_loss
Action Input: 2023-12-01, 2025-09-01Purchasing power change for all provinces between 2023-12-01 and 2025-09-01:
jurisdiction
Buenos Aires                      0.483322
Catamarca                         4.907918
Chaco                            33.482658
Chubut                           17.669821
Ciudad de Buenos Aires           -3.633964
Córdoba                           3.878260
Corrientes                       12.090993
Entre Ríos                       -0.667709
Formosa                          20.482779
Jujuy                             3.309000
La Pampa                        -20.009583
L

In [4]:
# Question 2
q2 = "Which are the three top paying and low paying provinces right now?"
response2 = journalist.query(q2)
print(response2['output'])



> Entering new AgentExecutor chain...
Thought: I need to find the three highest and three lowest paying provinces based on the most recent salary data available (up to 2025-09-01). Let me first check the latest date in the data and then get the current nominal salaries for all provinces.

Action: python_repl_ast
Action Input: print(df1.tail())jurisdiction  Buenos Aires     Catamarca        Chaco       Chubut  \
2024-09-01     574942.6250  458379.09375  716312.1875  572235.2500   
2024-12-01     619154.3125  498667.62500  803199.1250  637279.7500   
2025-03-01     675081.5625  549991.62500  890694.1250  684325.1875   
2025-06-01     719490.2500  594013.62500  967025.3125  740549.3750   
2025-09-01     781326.3750  696583.68750  967025.3125  781322.0000   

jurisdiction  Ciudad de Buenos Aires      Córdoba  Corrientes   Entre Ríos  \
2024-09-01               684297.5000  711362.6875   620082.50  563162.0000   
2024-12-01               754874.8125  785966.5000   811332.50  584877.7500  

KeyboardInterrupt: 

In [7]:
# Question 3
q3 = "How has the real salary from Buenos Aires changed in the last 10 years?"
response3 = journalist.query(q3)
print(response3['output'])



> Entering new AgentExecutor chain...
Thought: I need to check the real salaries for Buenos Aires over the last 10 years to analyze the changes.
Action: python_repl_ast
Action Input: print(df2['Buenos Aires'].tail(10))2023-06-01    10750.967773
2023-09-01    11970.545898
2023-12-01     8286.025391
2024-03-01     8209.655273
2024-06-01     7767.097656
2024-09-01     8072.494629
2024-12-01     8047.228027
2025-03-01     8081.600098
2025-06-01     8124.721680
2025-09-01     8326.073242
Name: Buenos Aires, dtype: float32
I need to analyze the trend of real salaries for Buenos Aires over the last 10 years based on the data I just retrieved.

The last 10 entries for real salaries in Buenos Aires are as follows:

- 2023-06-01: 10,750.97 AR$
- 2023-09-01: 11,970.55 AR$
- 2023-12-01: 8,286.03 AR$
- 2024-03-01: 8,209.66 AR$
- 2024-06-01: 7,767.10 AR$
- 2024-09-01: 8,072.49 AR$
- 2024-12-01: 8,047.23 AR$
- 2025-03-01: 8,081.60 AR$
- 2025-06-01: 8,124.72 AR$
- 2025-09-01: 8,326.07 AR$

From this